In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# Setting backend for matplotlib to 'Agg' to avoid display issues
plt.switch_backend('Agg')

print("Project: Ethereum Fraud Detection with Explainable AI")

# 01. Data Loading
df = pd.read_csv('./dataset/transaction_dataset.csv')

# 02. Data Cleaning and Preprocessing (Corrected)
# Handling duplicate columns
if df.columns.duplicated().any():
    df = df.loc[:, ~df.columns.duplicated()]

# Add 'Unnamed: 0' to the list of columns to drop.
# These columns are identifiers, not predictive features.
cols_to_drop = ['Unnamed: 0', 'Index', 'Address', 'ERC20 most sent token type', 'ERC20_most_rec_token_type']
df = df.drop(columns=cols_to_drop, errors='ignore')

# Find and convert any remaining object columns
obj_cols = df.select_dtypes(include=['object']).columns
if not obj_cols.empty:
    for col in obj_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Filling NaN values
df = df.fillna(0)

# 03. Feature Engineering and Splitting
print("\n03. Feature Engineering and Splitting")
X = df.drop(columns=['FLAG'])
y = df['FLAG']

print("\nTarget Variable Distribution (FLAG):")
print(y.value_counts(normalize=True))

# Spliting data, stratifying to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTraining data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# 04. Model Training (RandomForestClassifier)
print("\n04. Model Training (RandomForest)")

# Using class_weight='balanced' to automatically handle the imbalanced dataset
model = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=42, 
    # Usage all available cores
    n_jobs=-1  
)

print("Training the RandomForest model...")
model.fit(X_train, y_train)
print("Model training complete.")

# 05. Evaluating Model
y_pred = model.predict(X_test)

print("\nClassification Report (Realistic):")
report = classification_report(y_test, y_pred, target_names=['Not Fraud (0)', 'Fraud (1)'])
print(report)

print("\nConfusion Matrix (Realistic):")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Plotting Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, xticklabels=['Predicted Not Fraud', 'Predicted Fraud'], yticklabels=['Actual Not Fraud', 'Actual Fraud'])

plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix_heatmap.png')
print("\nConfusion matrix heatmap saved as 'confusion_matrix_heatmap.png'")

# 06. Explainable AI (XAI) with Feature Importance
print("\n06. Explainable AI (Global Feature Importance)")

importances = model.feature_importances_
feature_names = X_train.columns

# Creating a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n10 Most Important Features (Realistic):")
print(importance_df.head(10))

# Plotting the top 10 features
plt.figure(figsize=(10, 8))
top_10_features = importance_df.head(10)
plt.barh(top_10_features['Feature'], top_10_features['Importance'])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("10 Most Important Features (RandomForest)")
# Displaying the most important feature at the top
plt.gca().invert_yaxis()  
plt.tight_layout()
plt.savefig('feature_importance_plot.png')
print("\nFeature importance plot saved as 'feature_importance_plot.png'")

Project: Ethereum Fraud Detection with Explainable AI

03. Feature Engineering and Splitting

Target Variable Distribution (FLAG):
FLAG
0    0.778579
1    0.221421
Name: proportion, dtype: float64

Training data shape: (7872, 47)
Testing data shape: (1969, 47)

04. Model Training (RandomForest)
Training the RandomForest model...
Model training complete.

Classification Report (Realistic):
               precision    recall  f1-score   support

Not Fraud (0)       0.96      0.99      0.97      1533
    Fraud (1)       0.97      0.84      0.90       436

     accuracy                           0.96      1969
    macro avg       0.96      0.92      0.94      1969
 weighted avg       0.96      0.96      0.96      1969


Confusion Matrix (Realistic):
[[1522   11]
 [  68  368]]

Confusion matrix heatmap saved as 'confusion_matrix_heatmap.png'

06. Explainable AI (Global Feature Importance)

10 Most Important Features (Realistic):
                                              Feature  Importa

WITH WHITE-BOX MODEL AND XAI(FEATURE IMPORTANCE)

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

# Setting backend for matplotlib to 'Agg' to avoid display issues
plt.switch_backend('Agg')

print("Project: Ethereum Fraud Detection with Explainable AI - Logistic Regression")

# 01. Data Loading
df = pd.read_csv('./dataset/transaction_dataset.csv')

# 02. Data Cleaning and Preprocessing (Corrected)
# Handling duplicate columns
if df.columns.duplicated().any():
    df = df.loc[:, ~df.columns.duplicated()]

# Drop non-predictive columns
cols_to_drop = ['Unnamed: 0', 'Index', 'Address', 'ERC20 most sent token type', 'ERC20_most_rec_token_type']
df = df.drop(columns=cols_to_drop, errors='ignore')

# Convert remaining object columns to numeric, coercing errors to NaN
obj_cols = df.select_dtypes(include=['object']).columns
if not obj_cols.empty:
    for col in obj_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill NaN values with 0
df = df.fillna(0)

# 03. Feature Engineering and Splitting
print("\n03. Feature Engineering and Splitting")
X = df.drop(columns=['FLAG'])
y = df['FLAG']

print("\nTarget Variable Distribution (FLAG):")
print(y.value_counts(normalize=True))

# Spliting data with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# 04. Model Training (Logistic Regression)
print("\n04. Model Training (Logistic Regression)")

logreg_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print("Training the Logistic Regression model...")
logreg_model.fit(X_train, y_train)
print("Model training complete.")

# 05. Evaluating Model
y_pred = logreg_model.predict(X_test)

print("\nClassification Report (Logistic Regression):")
report = classification_report(y_test, y_pred, target_names=['Not Fraud (0)', 'Fraud (1)'])
print(report)

print("\nConfusion Matrix (Logistic Regression):")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Plotting Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', cbar=True,
    xticklabels=['Predicted Not Fraud', 'Predicted Fraud'],
    yticklabels=['Actual Not Fraud', 'Actual Fraud']
)
plt.title('Confusion Matrix Heatmap (Logistic Regression)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix_heatmap_logreg.png')
print("\nConfusion matrix heatmap saved as 'confusion_matrix_heatmap_logreg.png'")

# 06. Explainable AI (Global Feature Importance - Coefficients)
print("\n06. Explainable AI (Feature Coefficients)")

feature_names = X_train.columns
coefficients = logreg_model.coef_[0]

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', ascending=False)

print("\n10 Positively Correlated Features (Fraud Risk Increasing):")
print(importance_df.head(10))

print("\n10 Negatively Correlated Features (Fraud Risk Decreasing):")
print(importance_df.tail(10))

# Plotting the top 10 positive and negative features
plt.figure(figsize=(12, 8))
top_pos = importance_df.head(10)
top_neg = importance_df.tail(10)
top_features = pd.concat([top_pos, top_neg])

colors = ['green' if c > 0 else 'red' for c in top_features['Coefficient']]
plt.barh(top_features['Feature'], top_features['Coefficient'], color=colors)
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.title("10 Positive and Negative Feature Coefficients (Logistic Regression)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_coefficients_logreg.png')
print("\nFeature coefficients plot saved as 'feature_coefficients_logreg.png'")

Project: Ethereum Fraud Detection with Explainable AI - Logistic Regression

03. Feature Engineering and Splitting

Target Variable Distribution (FLAG):
FLAG
0    0.778579
1    0.221421
Name: proportion, dtype: float64

Training data shape: (7872, 47)
Testing data shape: (1969, 47)

04. Model Training (Logistic Regression)
Training the Logistic Regression model...
Model training complete.

Classification Report (Logistic Regression):
               precision    recall  f1-score   support

Not Fraud (0)       0.83      0.98      0.90      1533
    Fraud (1)       0.80      0.32      0.46       436

     accuracy                           0.83      1969
    macro avg       0.82      0.65      0.68      1969
 weighted avg       0.83      0.83      0.80      1969


Confusion Matrix (Logistic Regression):
[[1498   35]
 [ 297  139]]

Confusion matrix heatmap saved as 'confusion_matrix_heatmap_logreg.png'

06. Explainable AI (Feature Coefficients)

10 Positively Correlated Features (Fraud Ris

WITH BLACK_BOX MODEL and XAI(FEATURE IMPORTANCE --> PDP)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay

# Avoiding display issues on headless systems by setting bacckend for matplotlib as 'Agg'
plt.switch_backend('Agg')

def load_and_preprocess(filepath):
    df = pd.read_csv(filepath)

    # Removing duplicate columns if any
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()]

    # Droping non-predictive columns
    cols_to_drop = ['Unnamed: 0', 'Index', 'Address', 'ERC20 most sent token type', 'ERC20_most_rec_token_type']
    df = df.drop(columns=cols_to_drop, errors='ignore')

    # Convert object columns to numeric, coercing errors to NaN
    obj_cols = df.select_dtypes(include=['object']).columns
    if not obj_cols.empty:
        for col in obj_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Filling NaNs with '0'
    df = df.fillna(0)
    return df

def split_features_target(df, target_column='FLAG', test_size=0.2, random_state=42):
    X = df.drop(columns=[target_column])
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    return X_train, X_test, y_train, y_test

# Training 'RandomForestClassifier' with balanced class weights.
def train_random_forest(X_train, y_train, n_estimators=100, random_state=42):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        class_weight='balanced',
        random_state=random_state,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model

# Predicting and returning the resullts
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Not Fraud (0)', 'Fraud (1)']))

    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    # Ploting confusion matrix heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Not Fraud', 'Predicted Fraud'], yticklabels=['Actual Not Fraud', 'Actual Fraud'])
    plt.title('Confusion Matrix Heatmap')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_heatmap.png')
    plt.clf()

def plot_feature_importance(model, feature_names, top_n=10):
    importances = model.feature_importances_
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print(f"\n{top_n} Most Important Features:")
    print(importance_df.head(top_n))

    plt.figure(figsize=(10, 8))
    top_features = importance_df.head(top_n)
    plt.barh(top_features['Feature'], top_features['Importance'])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title(f"Top {top_n} Feature Importances (RandomForest)")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('feature_importance_plot.png')
    plt.clf()

def plot_partial_dependence_plots(model, X, features, top_n=3):
    features = list(features[:top_n])
    print(f"\nGenerating Partial Dependence Plots for top {top_n} features: {features}")

    disp = PartialDependenceDisplay.from_estimator(
        model,
        X,
        features=features,
        grid_resolution=50,
        kind='average'
    )
    plt.tight_layout()
    plt.savefig('partial_dependence_plots.png')
    plt.clf()

def main():
    print("Ethereum Fraud Detection with Explainable AI")

    # Load and preprocess data
    df = load_and_preprocess('./dataset/etheruem_transaction_dataset.csv')

    print("\nTarget Variable Distribution:")
    print(df['FLAG'].value_counts(normalize=True))

    # Spliting dataset (80-20)
    X_train, X_test, y_train, y_test = split_features_target(df)

    print(f"\nTraining data shape: {X_train.shape}")
    print(f"Testing data shape: {X_test.shape}")

    # Training the model
    model = train_random_forest(X_train, y_train)

    # Evaluating the model
    evaluate_model(model, X_test, y_test)

    # Feature by importance
    plot_feature_importance(model, X_train.columns, top_n=10)

    # Partial Dependence Plots (for top 3 features)
    feature_importances = model.feature_importances_
    sorted_idx = np.argsort(feature_importances)[::-1]  # descending order
    top_features = X_train.columns[sorted_idx][:3]
    plot_partial_dependence_plots(model, X_test, top_features, top_n=3)

if __name__ == "__main__":
    main()

Project: Ethereum Fraud Detection with Explainable AI

Target Variable Distribution:
FLAG
0    0.778579
1    0.221421
Name: proportion, dtype: float64

Training data shape: (7872, 47)
Testing data shape: (1969, 47)

Classification Report:
               precision    recall  f1-score   support

Not Fraud (0)       0.96      0.99      0.97      1533
    Fraud (1)       0.97      0.84      0.90       436

     accuracy                           0.96      1969
    macro avg       0.96      0.92      0.94      1969
 weighted avg       0.96      0.96      0.96      1969


Confusion Matrix:
[[1522   11]
 [  68  368]]

10 Most Important Features:
                                              Feature  Importance
2             Time Diff between first and last (Mins)    0.109828
10                                   avg val received    0.076831
19                               total ether received    0.075679
1                        Avg min between received tnx    0.068202
18                     

WITH WHITE_BOX MODEL and XAI(FEATURE IMPORTANCE --> PDP)

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.inspection import PartialDependenceDisplay

# Avoiding display issues on headless systems by setting backend for matplotlib as 'Agg'
plt.switch_backend('Agg')

def load_and_preprocess(filepath):
    df = pd.read_csv(filepath)

    # Removing duplicate columns if any
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()]

    # Dropping non-predictive columns
    cols_to_drop = ['Unnamed: 0', 'Index', 'Address', 'ERC20 most sent token type', 'ERC20_most_rec_token_type']
    df = df.drop(columns=cols_to_drop, errors='ignore')

    # Convert object columns to numeric, coercing errors to NaN
    obj_cols = df.select_dtypes(include=['object']).columns
    if not obj_cols.empty:
        for col in obj_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Filling NaNs with '0'
    df = df.fillna(0)
    return df

def split_features_target(df, target_column='FLAG', test_size=0.2, random_state=42):
    X = df.drop(columns=[target_column])
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    return X_train, X_test, y_train, y_test

# Training Decision Tree with controlled depth for interpretability
def train_decision_tree(X_train, y_train, random_state=42):
    model = DecisionTreeClassifier(
        max_depth=15,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=random_state
    )
    model.fit(X_train, y_train)
    return model

# Predicting and returning the results
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Not Fraud (0)', 'Fraud (1)']))

    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    # Plotting confusion matrix heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Predicted Not Fraud', 'Predicted Fraud'], 
                yticklabels=['Actual Not Fraud', 'Actual Fraud'])
    plt.title('Confusion Matrix Heatmap')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_heatmap.png')
    plt.clf()

def plot_feature_importance(model, feature_names, top_n=10):
    # Decision Tree feature importances
    importances = model.feature_importances_
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print(f"\n{top_n} Most Important Features:")
    print(importance_df.head(top_n))

    plt.figure(figsize=(10, 8))
    top_features = importance_df.head(top_n)
    plt.barh(top_features['Feature'], top_features['Importance'], color='steelblue')
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title(f"Top {top_n} Feature Importances (Decision Tree)")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('feature_importance_plot.png')
    plt.clf()
    
    return importance_df

def plot_partial_dependence_plots(model, X, features, top_n=3):
    features = list(features[:top_n])
    print(f"\nGenerating Partial Dependence Plots for top {top_n} features: {features}")

    disp = PartialDependenceDisplay.from_estimator(
        model,
        X,
        features=features,
        grid_resolution=50,
        kind='average'
    )
    plt.tight_layout()
    plt.savefig('partial_dependence_plots.png')
    plt.clf()

def plot_tree_structure(model, feature_names, max_depth_to_show=3):
    """Visualize the top levels of the decision tree"""
    plt.figure(figsize=(20, 10))
    plot_tree(
        model,
        feature_names=feature_names,
        class_names=['Not Fraud', 'Fraud'],
        filled=True,
        rounded=True,
        max_depth=max_depth_to_show,
        fontsize=8
    )
    plt.title(f'Decision Tree Structure (Top {max_depth_to_show} Levels)')
    plt.tight_layout()
    plt.savefig('decision_tree_structure.png', dpi=300, bbox_inches='tight')
    plt.clf()

def main():
    print("Ethereum Fraud Detection with Explainable AI - Decision Tree")
    print("="*70)

    # Load and preprocess data
    df = load_and_preprocess('./dataset/etheruem_transaction_dataset.csv')

    print("\nTarget Variable Distribution:")
    print(df['FLAG'].value_counts(normalize=True))

    # Splitting dataset (80-20)
    X_train, X_test, y_train, y_test = split_features_target(df)

    print(f"\nTraining data shape: {X_train.shape}")
    print(f"Testing data shape: {X_test.shape}")

    # Training the model
    print("\nTraining Decision Tree model...")
    model = train_decision_tree(X_train, y_train)
    
    print(f"Tree depth: {model.get_depth()}")
    print(f"Number of leaves: {model.get_n_leaves()}")

    # Evaluating the model
    evaluate_model(model, X_test, y_test)

    # Feature importance
    importance_df = plot_feature_importance(model, X_train.columns, top_n=10)

    # Partial Dependence Plots (for top 3 features)
    top_features = importance_df.head(3)['Feature'].values
    plot_partial_dependence_plots(model, X_test, top_features, top_n=3)
    
    # Visualizing tree structure
    plot_tree_structure(model, X_train.columns, max_depth_to_show=3)
    
    print("Analysis complete!")

if __name__ == "__main__":
    main()

Ethereum Fraud Detection with Explainable AI - Decision Tree

Target Variable Distribution:
FLAG
0    0.778579
1    0.221421
Name: proportion, dtype: float64

Training data shape: (7872, 47)
Testing data shape: (1969, 47)

Training Decision Tree model...
Tree depth: 15
Number of leaves: 164

Classification Report:
               precision    recall  f1-score   support

Not Fraud (0)       0.96      0.93      0.95      1533
    Fraud (1)       0.78      0.87      0.82       436

     accuracy                           0.92      1969
    macro avg       0.87      0.90      0.89      1969
 weighted avg       0.92      0.92      0.92      1969


Confusion Matrix:
[[1429  104]
 [  57  379]]

10 Most Important Features:
                                    Feature  Importance
19                     total ether received    0.308464
2   Time Diff between first and last (Mins)    0.203683
6            Unique Received From Addresses    0.109346
8                        min value received    0.076

c:\Users\siddh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\inspection\_partial_dependence.py:717: FutureWarning: The column 6 contains integer data. Partial dependence plots are not supported for integer data: this can lead to implicit rounding with NumPy arrays or even errors with newer pandas versions. Please convert numerical featuresto floating point dtypes ahead of time to avoid problems. This will raise ValueError in scikit-learn 1.9.
  warnings.warn(


Analysis complete!
